# Attention Variants: MHA vs GQA

Compare multi-head attention (MHA) and grouped-query attention
(GQA) by visualizing attention maps and parameter counts.
Uses the analysis toolkit to extract weights from the fused
attention path.

**Prerequisites:** `uv sync --extra dev --extra analysis`

In [1]:
import mlx.core as mx

from lmxlab.analysis.attention import extract_attention_maps
from lmxlab.models.base import LanguageModel
from lmxlab.models.gpt import gpt_tiny
from lmxlab.models.llama import llama_tiny

/Users/michaelellis/Projects/lmt-metal/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## MHA: Multi-Head Attention (GPT)

Standard multi-head attention
([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)):
every head has its own Q, K, V projections. All heads have
independent key/value pairs.

In [2]:
gpt_cfg = gpt_tiny()
gpt = LanguageModel(gpt_cfg)
mx.eval(gpt.parameters())

tokens = mx.array([[0, 1, 2, 3, 4, 5, 6, 7]], dtype=mx.int32)
maps = extract_attention_maps(gpt, tokens)

print(f"GPT: {gpt_cfg.block.n_heads} heads, MHA")
print(f"Attention maps: {list(maps.keys())}")
print(f"Shape per layer: {maps['layer_0'].shape}")
print(f"  = (batch, heads, seq, seq)")

GPT: 2 heads, MHA
Attention maps: ['layer_0', 'layer_1']
Shape per layer: (1, 2, 8, 8)
  = (batch, heads, seq, seq)


In [3]:
# Verify causal masking: position i can only attend to positions <= i
w = maps["layer_0"]
mx.eval(w)
print("Attention weights for head 0, layer 0:")
for i in range(8):
    row = [f"{w[0, 0, i, j].item():.2f}" for j in range(8)]
    print(f"  pos {i}: [{', '.join(row)}]")

Attention weights for head 0, layer 0:
  pos 0: [1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00]
  pos 1: [0.65, 0.35, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00]
  pos 2: [0.29, 0.37, 0.35, 0.00, 0.00, 0.00, 0.00, 0.00]
  pos 3: [0.18, 0.16, 0.41, 0.26, 0.00, 0.00, 0.00, 0.00]
  pos 4: [0.17, 0.22, 0.20, 0.18, 0.23, 0.00, 0.00, 0.00]
  pos 5: [0.22, 0.24, 0.17, 0.12, 0.15, 0.11, 0.00, 0.00]
  pos 6: [0.19, 0.14, 0.15, 0.07, 0.18, 0.13, 0.14, 0.00]
  pos 7: [0.08, 0.15, 0.13, 0.09, 0.08, 0.10, 0.19, 0.16]


## GQA: Grouped Query Attention (LLaMA)

GQA ([Ainslie et al., 2023](https://arxiv.org/abs/2305.13245))
uses fewer KV heads than Q heads. Multiple Q heads share the
same K/V pair. This reduces KV cache memory proportionally
while maintaining model quality.

In [4]:
llama_cfg = llama_tiny()
llama = LanguageModel(llama_cfg)
mx.eval(llama.parameters())

maps_gqa = extract_attention_maps(llama, tokens)

n_q = llama_cfg.block.n_heads
n_kv = llama_cfg.block.effective_n_kv_heads
ratio = n_q // n_kv

print(f"LLaMA: {n_q} Q heads, {n_kv} KV heads (ratio {ratio}:1)")
print(f"Shape: {maps_gqa['layer_0'].shape}")
print(f"KV cache savings: {ratio}x smaller than MHA")

LLaMA: 4 Q heads, 2 KV heads (ratio 2:1)
Shape: (1, 4, 8, 8)
KV cache savings: 2x smaller than MHA


## Parameter Efficiency Comparison

GQA saves parameters in K/V projections while keeping
the same number of Q heads for expressiveness
(see Table 1 in [Ainslie et al., 2023](https://arxiv.org/abs/2305.13245)).

In [5]:
d = 64  # d_model for tiny configs

# MHA: Q, K, V, O all d×d
mha_params = 4 * d * d

# GQA: Q and O are d×d, K and V are d×(d/ratio)
gqa_kv_dim = n_kv * (d // n_q)
gqa_params = 2 * d * d + 2 * d * gqa_kv_dim

savings = (1 - gqa_params / mha_params) * 100
print(f"MHA attention params: {mha_params:,}")
print(f"GQA attention params: {gqa_params:,}")
print(f"Savings: {savings:.0f}%")

MHA attention params: 16,384
GQA attention params: 12,288
Savings: 25%


## Visualize Attention (requires matplotlib)

In [6]:
try:
    from lmxlab.analysis.plotting import plot_attention_heatmap

    fig = plot_attention_heatmap(maps["layer_0"], head=0, layer=0)
    fig.suptitle("GPT MHA — Layer 0, Head 0")
    fig.show()
except ImportError:
    print("Install matplotlib: uv sync --extra analysis")

/var/folders/b3/kr49kw115k73d4m4rh1lpc400000gn/T/ipykernel_42824/3708698420.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()


## References

- Vaswani et al. (2017). [Attention Is All You Need](https://arxiv.org/abs/1706.03762). NeurIPS 2017.
- Ainslie et al. (2023). [GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245). EMNLP 2023.
- Touvron et al. (2023). [LLaMA: Open and Efficient Foundation Language Models](https://arxiv.org/abs/2302.13971). arXiv preprint.